In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt

os.makedirs('data/cleaned', exist_ok=True)

**CLEAN BOUNDARIES**

In [2]:
boundaries_raw = pd.read_csv('../data/raw/Boundaries_-_Community_Areas_20260305.csv')

print(f"Shape: {boundaries_raw.shape}")
boundaries_raw.info()
print(boundaries_raw.drop(columns=['the_geom']).head(3).to_string())

Shape: (77, 6)
<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   the_geom    77 non-null     str  
 1   AREA_NUMBE  77 non-null     int64
 2   COMMUNITY   77 non-null     str  
 3   AREA_NUM_1  77 non-null     int64
 4   SHAPE_AREA  77 non-null     str  
 5   SHAPE_LEN   77 non-null     str  
dtypes: int64(2), str(4)
memory usage: 1.9 MB
   AREA_NUMBE    COMMUNITY  AREA_NUM_1       SHAPE_AREA       SHAPE_LEN
0           1  ROGERS PARK           1  51,259,902.4506  34,052.3975757
1           2   WEST RIDGE           2  98,429,094.8621  43,020.6894583
2           3       UPTOWN           3  65,095,642.7289  46,972.7945549


In [3]:
boundaries = boundaries_raw[['the_geom', 'AREA_NUMBE', 'COMMUNITY']].copy()

# Rename to match project
boundaries = boundaries.rename(columns={
    'AREA_NUMBE': 'area_number',
    'COMMUNITY':  'community_area'
})

# Normalize ALL CAPS to Title Case so it matches listings and income
boundaries['community_area'] = boundaries['community_area'].str.title()

# Fix known casing issues that str.title() gets wrong
boundaries['community_area'] = boundaries['community_area'].replace({
    'Mckinley Park': 'McKinley Park',
    'Ohare':         "O'Hare"
})

# Parse WKT geometry string into actual geometry objects
boundaries['geometry'] = boundaries['the_geom'].apply(wkt.loads)

# Convert to GeoDataFrame
boundaries = gpd.GeoDataFrame(boundaries, geometry='geometry', crs='EPSG:4326')

# Drop raw WKT column — no longer needed
boundaries = boundaries.drop(columns=['the_geom'])

print(f"Shape: {boundaries.shape}")
print(boundaries[['area_number', 'community_area']].head(5).to_string())
print(f"\nCRS: {boundaries.crs}")          # expect EPSG:4326
print(f"Nulls:\n{boundaries.isnull().sum()}")

Shape: (77, 3)
   area_number  community_area
0            1     Rogers Park
1            2      West Ridge
2            3          Uptown
3            4  Lincoln Square
4            5    North Center

CRS: EPSG:4326
Nulls:
area_number       0
community_area    0
geometry          0
dtype: int64


In [4]:
# This is critical — if names don't match here they won't match in the final merge either
income = pd.read_csv('../data/cleaned/cleaned_income.csv')

unmatched = set(boundaries['community_area']) - set(income['community_area'])
print(f"Unmatched boundary names: {len(unmatched)}  (expect 0)")
if unmatched:
    print("Fix these manually:", unmatched)

Unmatched boundary names: 0  (expect 0)


**CRIME DATA**

In [5]:
crime = pd.read_csv('../data/cleaned/cleaned_crime.csv')

print(crime.columns.tolist())
print(crime.dtypes)
print(crime.head(3))

print(f"Shape: {crime.shape}")
print(crime.dtypes)
print(f"\nNull lat/lon: {crime[['latitude','longitude']].isnull().sum()}")
print(crime.head(3).to_string())

['crime_id', 'date', 'primary_type', 'secondary_type', 'location_description', 'arrest', 'domestic', 'beat', 'latitude', 'longitude']
crime_id                    str
date                        str
primary_type                str
secondary_type              str
location_description        str
arrest                     bool
domestic                   bool
beat                      int64
latitude                float64
longitude               float64
dtype: object
   crime_id                 date     primary_type           secondary_type  \
0  JJ189129  2025-03-19 13:00:00  criminal damage               to vehicle   
1  JJ193644  2025-03-24 18:38:00    other offense  harassment by telephone   
2  JJ343927  2025-07-22 08:15:00          battery                   simple   

  location_description  arrest  domestic  beat   latitude  longitude  
0               STREET   False     False  1414  41.920038 -87.698673  
1            APARTMENT   False     False  1412  41.934109 -87.712348  
2     

In [6]:
# Drop any remaining null coordinates otherwise can't spacial join
before = len(crime)
crime = crime.dropna(subset=['latitude', 'longitude'])
after = len(crime)
print(f"Dropped {before - after} rows with null coordinates")
print(f"Remaining: {after}")

# Convert to GeoDataFrame using lat/lon columns
crime_gdf = gpd.GeoDataFrame(
    crime,
    geometry=gpd.points_from_xy(crime['longitude'], crime['latitude']),
    crs='EPSG:4326'
)

print(f"\nCrime GeoDataFrame shape: {crime_gdf.shape}")
print(f"CRS: {crime_gdf.crs}")  # must match boundaries CRS — both EPSG:4326

Dropped 0 rows with null coordinates
Remaining: 188734

Crime GeoDataFrame shape: (188734, 11)
CRS: EPSG:4326


**SPACIAL JOIN**

In [7]:
# Assigns each crime the community area whose polygon contains it
# how='left' keeps all crimes even if they fall outside all boundaries
# predicate='within' means the point must be inside the polygon
crime_by_area = gpd.sjoin(
    crime_gdf,
    boundaries[['area_number', 'community_area', 'geometry']],
    how='left',
    predicate='within'
)

# Crimes that fell outside all boundaries (lake, highways, etc.)
outside = crime_by_area['community_area'].isnull().sum()
print(f"Crimes outside all boundaries: {outside}")
print(f"  ({outside / len(crime_by_area) * 100:.1f}% of total get dropped)")

# Drop unmatched
crime_with_area = crime_by_area.dropna(subset=['community_area'])
print(f"\nCrimes successfully assigned to an area: {len(crime_by_area)}")

Crimes outside all boundaries: 587
  (0.3% of total get dropped)

Crimes successfully assigned to an area: 188734


**AGGREGATE**

In [8]:
violent_types = [
    'battery', 'assault', 'robbery', 'homicide',
    'criminal sexual assault', 'kidnapping', 'arson',
    'stalking', 'intimidation', 'offense involving children',
    'human trafficking', 'weapons violation'
]

crime_by_area = crime_by_area.groupby('community_area').agg(
    area_number    = ('area_number',  'first'),
    total_crimes   = ('crime_id',     'count'),
    violent_crimes = ('primary_type', lambda x: x.isin(violent_types).sum()),
    theft_count    = ('primary_type', lambda x: (x == 'theft').sum()),
    arrest_rate    = ('arrest',       'mean'),
).reset_index()

crime_by_area['pct_violent'] = (
    crime_by_area['violent_crimes'] / crime_by_area['total_crimes']
).round(4)

print(f"Shape: {crime_by_area.shape}")   # expect (77, 7) or fewer if some areas had 0 crimes
print(crime_by_area.sort_values('total_crimes', ascending=False).head(5).to_string())

Shape: (77, 7)
     community_area  area_number  total_crimes  violent_crimes  theft_count  arrest_rate  pct_violent
47  Near North Side          8.0         10394            2262         4653     0.148259       0.2176
49   Near West Side         28.0          9235            2208         2953     0.108284       0.2391
41             Loop         32.0          8126            1810         3276     0.198499       0.2227
5            Austin         25.0          8106            2448         1571     0.226869       0.3020
75        West Town         24.0          6312            1163         2180     0.078897       0.1843


**ZERO CRIMES**

In [9]:
# Some community areas may have zero crimes and won't appear after groupby
# We need all 77 areas for a clean join — fill missing with 0
all_areas = boundaries[['area_number', 'community_area']]
crime_by_area = all_areas.merge(crime_by_area, on=['area_number', 'community_area'], how='left')
crime_by_area[['total_crimes', 'violent_crimes', 'theft_count']] = (
    crime_by_area[['total_crimes', 'violent_crimes', 'theft_count']].fillna(0).astype(int)
)
fill_cols = ['total_crimes', 'violent_crimes', 'theft_count', 
             'arrest_rate', 'pct_violent']
crime_by_area[fill_cols] = crime_by_area[fill_cols].fillna(0)

print(f"Shape: {crime_by_area.shape}")   # expect (77, 7)
print(f"Nulls:\n{crime_by_area.isnull().sum()}")

Shape: (77, 7)
Nulls:
area_number       0
community_area    0
total_crimes      0
violent_crimes    0
theft_count       0
arrest_rate       0
pct_violent       0
dtype: int64


**VALIDATION**

In [10]:
assert len(crime_by_area) == 77, \
    f"Expected 77 rows, got {len(crime_by_area)}"

assert crime_by_area.isnull().sum().sum() == 0, \
    "Unexpected nulls in crime_by_area"

assert crime_by_area['total_crimes'].min() >= 0, \
    "Negative crime counts found"

assert crime_by_area['pct_violent'].between(0, 1).all(), \
    "pct_violent outside 0-1 range"

assert crime_by_area['arrest_rate'].between(0, 1).all(), \
    "arrest_rate outside 0-1 range"

print("✓ All assertions passed")
print(f"\nTop 5 areas by total crimes:")
print(crime_by_area.nlargest(5, 'total_crimes')[
    ['community_area', 'total_crimes', 'violent_crimes', 'pct_violent']
])

✓ All assertions passed

Top 5 areas by total crimes:
     community_area  total_crimes  violent_crimes  pct_violent
7   Near North Side         10394            2262       0.2176
27   Near West Side          9235            2208       0.2391
31             Loop          8126            1810       0.2227
24           Austin          8106            2448       0.3020
23        West Town          6312            1163       0.1843


In [11]:
crime_by_area.to_csv('../data/cleaned/crime_by_area.csv', index=False)

print(f"Shape:   {crime_by_area.shape}")
print(f"Columns: {crime_by_area.columns.tolist()}")
print(f"\nCrime stats:")
print(crime_by_area[['total_crimes','violent_crimes','theft_count','arrest_rate','pct_violent']].describe().round(3))
print("\n✓ Saved to data/cleaned/crime_by_area.csv")

Shape:   (77, 7)
Columns: ['area_number', 'community_area', 'total_crimes', 'violent_crimes', 'theft_count', 'arrest_rate', 'pct_violent']

Crime stats:
       total_crimes  violent_crimes  theft_count  arrest_rate  pct_violent
count        77.000          77.000       77.000       77.000       77.000
mean       2443.468         630.623      660.169        0.139        0.252
std        2072.796         544.328      761.370        0.060        0.060
min         195.000          32.000       39.000        0.050        0.100
25%         996.000         219.000      253.000        0.102        0.209
50%        1872.000         440.000      455.000        0.128        0.249
75%        3078.000         913.000      739.000        0.157        0.292
max       10394.000        2448.000     4653.000        0.331        0.445

✓ Saved to data/cleaned/crime_by_area.csv


In [12]:
listings = pd.read_csv('../data/cleaned/cleaned_listings.csv')
income   = pd.read_csv('../data/cleaned/cleaned_income.csv')
crime    = pd.read_csv('../data/cleaned/crime_by_area.csv')

print(set(listings['community_area'].unique()) - set(income['community_area']))
print(set(listings['community_area'].unique()) - set(crime['community_area']))

{'Ohare', 'Mckinley Park'}
{'Ohare', 'Mckinley Park'}
